# Visualize existing cafl4ds runs

This notebook makes the current experiment outputs inspectable before larger experiments are run. It plots:

- frozen-probe accuracy (`linear_acc` and `knn_acc`),
- effective representation rank (`rankme` and, when present, `rankme_proj`), and
- representation drift (`cosine_drift` and `cka_drift`, including projector surfaces when present).

Curves remain separate for every seed and arm. Dashed vertical lines and labels mark changes in the stream regime. The default input is the newest complete getting-started run, but `RUN_ROOT` can point at any compatible run directory.

> Accuracy is a canary measurement that uses labels for evaluation. RankMe and drift are label-free health signals. A moving signal is evidence of change, not proof that the change caused an accuracy gain or loss.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display as show

plt.style.use("seaborn-v0_8-whitegrid")

## Configuration

Leave `RUN_ROOT` as `None` to use the newest health corpus, including P1.2 adaptation runs. Set it to a path such as `REPO_ROOT / "outputs" / "adaptation" / "20260917-143304"` to pin a run.

In [ ]:
def find_repo_root(start: Path) -> Path:
    """Locate the repository from a notebook or repository working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "cafl4ds").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the cafl4ds repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RUN_ROOT: Path | None = None
SAVE_FIGURE = True
FIGURE_PATH = REPO_ROOT / "outputs" / "visualizations" / "existing_runs.png"
CONDITION_FIGURE_PATH = REPO_ROOT / "outputs" / "visualizations" / "condition_accuracy.png"


def health_path(run_root: Path) -> Path | None:
    """Return a standard deployment or study health-corpus path."""
    candidates = [run_root / "corpus" / "health.csv", run_root / "deploy" / "corpus" / "health.csv"]
    return next((path for path in candidates if path.exists()), None)


def newest_complete_run() -> Path:
    """Return the most recently modified run with a standard health corpus."""
    roots = [
        REPO_ROOT / "outputs" / "getting-started",
        REPO_ROOT / "outputs" / "adaptation",
        REPO_ROOT / "outputs" / "adaptation-bdd",
    ]
    candidates = [path for root in roots if root.exists() for path in root.iterdir() if health_path(path)]
    if not candidates:
        raise FileNotFoundError(
            "No health corpus found below outputs/getting-started, outputs/adaptation, or outputs/adaptation-bdd."
        )
    return max(candidates, key=lambda path: health_path(path).stat().st_mtime)


RUN_ROOT = Path(RUN_ROOT).resolve() if RUN_ROOT is not None else newest_complete_run()
RUN_ROOT

## Load and normalize health-corpus and harness records

Deployment and P1.2 write a canonical `health.csv`. Older synthetic comparison runs also write one JSONL file per arm. Both forms are normalized to one table and assigned a trace identifier containing the study/source, backbone, arm, and seed.

In [ ]:
def load_deployment(path: Path) -> pd.DataFrame:
    """Load the canonical deployment health corpus."""
    frame = pd.read_csv(path)
    manifest_path = path.parent / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    frame["source"] = manifest.get("study", "deployment")
    frame["era"] = frame["era_index"]
    return frame


def parse_run_name(name: str) -> tuple[str, str, int]:
    """Infer backbone, arm, and seed metadata from a harness filename."""
    backbone = name.split("_", maxsplit=1)[0]
    arm = "live" if name.endswith("_live") else "positive_control" if name.endswith("_pc") else name
    seed_match = re.search(r"(?:^|_)s(\d+)(?:_|$)", name)
    seed = int(seed_match.group(1)) if seed_match else 0
    return backbone, arm, seed


def load_harness(path: Path) -> pd.DataFrame:
    """Load health-series records from one synthetic harness JSONL file."""
    records = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            if record.get("series") == "health":
                records.append(record)
    if not records:
        return pd.DataFrame()
    frame = pd.DataFrame.from_records(records)
    backbone, arm, seed = parse_run_name(path.stem)
    frame["source"] = "synthetic"
    frame["backbone"] = backbone
    frame["arm"] = arm
    frame["seed"] = seed
    frame["era_name"] = frame["era"].map(lambda value: f"regime{int(value)}")
    return frame


HEALTH_PATH = health_path(RUN_ROOT)
if HEALTH_PATH is None:
    raise FileNotFoundError(f"No health.csv found below {RUN_ROOT}")
parts = [load_deployment(HEALTH_PATH)]
harness_dir = RUN_ROOT / "harness"
if harness_dir.exists():
    parts.extend(load_harness(path) for path in sorted(harness_dir.glob("*.jsonl")))
health = pd.concat([part for part in parts if not part.empty], ignore_index=True, sort=False)
health["step"] = pd.to_numeric(health["step"])
health["seed"] = health["seed"].fillna(0).astype(int)
health["trace_id"] = health.apply(
    lambda row: f"{row['source']} · {row['backbone']} · {row['arm']} · seed {row['seed']}", axis=1
)
health = health.sort_values(["trace_id", "step"]).reset_index(drop=True)

print(f"Loaded {len(health)} health checkpoints from {RUN_ROOT}")
show(
    health.groupby(["source", "backbone", "arm", "seed"], dropna=False)
    .agg(checkpoints=("step", "size"), first_step=("step", "min"), last_step=("step", "max"))
    .reset_index()
)
show(health.head())

## Plot health over stream time

Deployment and synthetic runs use separate columns because their time horizons differ. Line color identifies a seed/arm trace; line style identifies the measurement or representation surface. Regime boundaries use the longest trace in each column as the reference schedule. When a stream changes regimes frequently, every boundary remains visible but only a sample of labels is drawn.

In [ ]:
def transition_schedule(frame: pd.DataFrame) -> list[tuple[float, str]]:
    """Return the first step and name of every contiguous regime segment."""
    ordered = frame.sort_values("step").drop_duplicates("step")
    changed = ordered["era"].ne(ordered["era"].shift())
    return [(float(row.step), str(row.era_name)) for row in ordered.loc[changed].itertuples()]


def add_regime_boundaries(ax: plt.Axes, frame: pd.DataFrame) -> None:
    """Draw all regime boundaries and a readable sample of their labels."""
    trace_sizes = frame.groupby("trace_id").size().sort_values(ascending=False)
    reference = frame[frame["trace_id"] == trace_sizes.index[0]]
    schedule = transition_schedule(reference)
    label_stride = max(1, (len(schedule) + 9) // 10)
    for index, (step, name) in enumerate(schedule):
        if index:
            ax.axvline(step, color="0.35", linestyle=":", linewidth=1.2, alpha=0.8)
        if index % label_stride == 0 or index == len(schedule) - 1:
            ax.text(
                step,
                0.985,
                name,
                transform=ax.get_xaxis_transform(),
                rotation=35,
                ha="left",
                va="top",
                fontsize=7,
                color="0.3",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.65, "pad": 0.5},
            )


def plot_family(ax: plt.Axes, frame: pd.DataFrame, metrics: dict[str, str], title: str, ylabel: str) -> None:
    """Plot related metrics while preserving a separate curve for every trace."""
    colors = plt.get_cmap("tab10")
    traces = list(frame["trace_id"].drop_duplicates())
    styles = ["-", "--", "-.", ":"]
    for trace_index, trace in enumerate(traces):
        subset = frame[frame["trace_id"] == trace].sort_values("step")
        for metric_index, (metric, label) in enumerate(metrics.items()):
            if metric not in subset or subset[metric].notna().sum() == 0:
                continue
            values = subset.dropna(subset=[metric])
            ax.plot(
                values["step"],
                values[metric],
                marker="o",
                markersize=3.5,
                linewidth=1.7,
                color=colors(trace_index % 10),
                linestyle=styles[metric_index % len(styles)],
                label=f"{trace} · {label}",
            )
    add_regime_boundaries(ax, frame)
    ax.set_title(title, loc="left", fontweight="bold", pad=10)
    ax.set_xlabel("Stream step")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, ncol=2, frameon=True)


sources = list(health["source"].drop_duplicates())
metric_rows = [
    ({"linear_acc": "linear probe", "knn_acc": "k-NN probe"}, "Frozen-probe accuracy", "Accuracy"),
    ({"rankme": "backbone", "rankme_proj": "projector"}, "RankMe", "Effective rank"),
    (
        {
            "cosine_drift": "cosine · backbone",
            "cosine_drift_proj": "cosine · projector",
            "cka_drift": "CKA · backbone",
            "cka_drift_proj": "CKA · projector",
        },
        "Representation drift",
        "Drift from reference",
    ),
]
fig, axes = plt.subplots(3, len(sources), figsize=(9 * len(sources), 15), squeeze=False, constrained_layout=True)
for column, source in enumerate(sources):
    source_frame = health[health["source"] == source]
    schedules = source_frame.groupby("trace_id").apply(transition_schedule, include_groups=False)
    if len({tuple(schedule) for schedule in schedules}) > 1:
        print(f"Note: {source} traces use different regime schedules; boundaries follow its longest trace.")
    for row, (metrics, title, ylabel) in enumerate(metric_rows):
        panel = chr(ord("A") + row)
        plot_family(axes[row, column], source_frame, metrics, f"{panel}. {title} — {source}", ylabel)
fig.suptitle(f"cafl4ds health trajectories — {RUN_ROOT.name}", fontsize=16, fontweight="bold")

if SAVE_FIGURE:
    FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_PATH, dpi=170, bbox_inches="tight")
    print(f"Saved {FIGURE_PATH}")

plt.show()

## Matched adaptation versus frozen by condition

P1.2 additionally reports `adapted − frozen` accuracy on the current regime and all earlier regimes. Positive values favor adaptation; negative values favor the init-matched frozen backbone. This section appears when the selected run contains `condition_accuracy.csv`.

In [ ]:
condition_path = RUN_ROOT / "condition_accuracy.csv"
if condition_path.exists():
    condition = pd.read_csv(condition_path)
    condition["is_current"] = condition["is_current"].astype(str).str.lower().eq("true")
    fig_conditions, condition_axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    colors = plt.get_cmap("tab10")
    for color_index, (seed, seed_frame) in enumerate(condition.groupby("seed")):
        color = colors(color_index % 10)
        current = seed_frame[seed_frame["is_current"]].sort_values("after_era")
        condition_axes[0].plot(current["after_era"], current["gain"], marker="o", color=color, label=f"seed {seed}")
        past = (
            seed_frame[seed_frame["eval_era"] < seed_frame["after_era"]]
            .groupby("after_era", as_index=False)["gain"]
            .mean()
        )
        condition_axes[1].plot(past["after_era"], past["gain"], marker="o", color=color, label=f"seed {seed}")
        final = seed_frame[seed_frame["after_era"] == seed_frame["after_era"].max()].sort_values("eval_era")
        condition_axes[2].plot(final["eval_era"], final["gain"], marker="o", color=color, label=f"seed {seed}")
    titles = ["Current-condition gain", "Earlier-condition mean gain", "Final gain by condition"]
    xlabels = ["Regime just completed", "Regime just completed", "Evaluated regime"]
    for axis, title, xlabel in zip(condition_axes, titles, xlabels, strict=True):
        axis.axhline(0.0, color="black", linewidth=1.0)
        axis.set_title(title, loc="left", fontweight="bold")
        axis.set_xlabel(xlabel)
        axis.set_ylabel("Accuracy gain: adapted − frozen")
        axis.legend()
    if SAVE_FIGURE:
        CONDITION_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
        fig_conditions.savefig(CONDITION_FIGURE_PATH, dpi=170, bbox_inches="tight")
        print(f"Saved {CONDITION_FIGURE_PATH}")
    plt.show()
else:
    print("No condition_accuracy.csv in this run; skipping the matched condition plots.")

## Reading the result

Use these plots to generate questions, not causal conclusions:

1. Look for accuracy changes close to regime transitions.
2. Check whether RankMe or drift changes before, with, or after the accuracy movement.
3. Compare arms within the same seed before comparing across seeds.
4. Treat synthetic smoke outputs as wiring evidence; the P1.2 corpus adds matched frozen arms and three seeds but still does not substitute for real BDD data.
5. On matched P1.2 runs, interpret gains relative to zero and check whether their sign repeats across seeds and conditions.